# Lab 7 — Large Language Models (LLMs)

**Name:** Maan Ghamdi  
**Student ID:** 2240001433

I explore the three transformer families through text generation, translation, sentiment classification, and summarization.

## The Three Main Transformer Model Families

| Family | Example Models | Main Use |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Classification, sentiment analysis, NER |
| **Encoder–decoder** | T5, BART | Translation, summarization, question answering |
| **Decoder-only** | GPT family | Text generation, conversation, completion |


## 1. Setup


I run all cells from top to bottom in a fresh Python 3.11 Jupyter kernel or a Google Colab Python runtime. The first run needs internet access to install the libraries and download the three public models. I use the CPU, so a GPU and an API key are not required. Downloads and the first run may take several minutes.

I pin Transformers and SentencePiece for consistent APIs. If I previously imported different versions in this session, I restart the kernel after installation and run all cells again.

In [1]:
%pip -q install "transformers==4.50.3" "sentencepiece==0.2.0" "torch>=2.6,<3"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import transformers
from transformers import pipeline, set_seed, AutoTokenizer, AutoModelForSeq2SeqLM

# A modest CPU thread count keeps these small examples responsive.
torch.set_num_threads(min(4, torch.get_num_threads()))
set_seed(42)
print("Libraries loaded successfully!")
print(f"Transformers: {transformers.__version__}; PyTorch: {torch.__version__}")

C:\Users\User\Documents\Codex\2026-11-03\pl\work\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded successfully!
Transformers: 4.50.3; PyTorch: 2.6.0+cpu


# Part A — Decoder-Only Model

## Decoder-only example: GPT-style Text Generation

A decoder-only model predicts the next token based on previous tokens.

We will use **DistilGPT-2**, a small GPT-style model.


In [3]:
generator = pipeline(
    "text-generation", model="distilgpt2", framework="pt", device=-1
)

prompt = "Natural language processing is"
output = generator(
    prompt,
    max_new_tokens=30,
    do_sample=False,
    pad_token_id=generator.tokenizer.eos_token_id,
)
print(output[0]["generated_text"])

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Device set to use cpu


Natural language processing is a common feature of the language processing process.























# Part B — Encoder–Decoder Model

## Encoder–decoder example: Instruction / Text-to-Text Generation

Encoder–decoder models contain two main parts:

1. The **encoder** reads the input.
2. The **decoder** produces the output.

We will use **FLAN-T5-small**.


In [4]:
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")
model.eval()

prompt = "Translate to French: I love natural language processing."
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
with torch.inference_mode():
    output = model.generate(**inputs, max_new_tokens=60, do_sample=False)
print(tokenizer.decode(output[0], skip_special_tokens=True))

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Je s'agissant de la technologie de linguage naturelle.


# Part C — Encoder-Only Model

## Encoder-only example: Sentiment Classification

Encoder-only models read the full input and create a representation of it.

They are very useful when the task is to **understand or classify text**.

Here we use a DistilBERT sentiment model.


In [5]:
classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    framework="pt",
    device=-1,
)

text = "This NLP lab is easy and interesting."
result = classifier(text, truncation=True)
print(result)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.999778687953949}]


# Tasks

Complete the following tasks.

## Task 1 — Model Families

For each task below, choose the most suitable model family:

1. Sentiment classification
2. Machine translation
3. Open-ended text generation
4. Named Entity Recognition
5. Text summarization

### My answers

| Task | Most suitable model family | My reason |
|---|---|---|
| Sentiment classification | Encoder-only | I use bidirectional context to classify the sentiment of the input. |
| Machine translation | Encoder–decoder | I encode the source sentence and decode a sentence in the target language. |
| Open-ended text generation | Decoder-only | I generate a continuation by predicting one next token at a time. |
| Named Entity Recognition | Encoder-only | I use surrounding context to assign entity labels to input tokens. |
| Text summarization | Encoder–decoder | I encode the source passage and generate a shorter text that captures its main ideas. |

These are typical choices for the lab's model families; other architectures can also perform some of these tasks.

## Task 2 — Encoder-only Model

Use the sentiment classifier on these three sentences:

```text
The movie was excellent.
The lecture was difficult to understand.
I enjoyed learning about transformers.
```

Print the predicted label and score for each sentence.

In [6]:
sentences = [
    "The movie was excellent.",
    "The lecture was difficult to understand.",
    "I enjoyed learning about transformers.",
]

predictions = classifier(sentences, truncation=True)
for sentence, prediction in zip(sentences, predictions):
    print(f"Sentence: {sentence}")
    print(f"Label: {prediction['label']} | Score: {prediction['score']:.4f}")
    print()

Sentence: The movie was excellent.
Label: POSITIVE | Score: 0.9999

Sentence: The lecture was difficult to understand.
Label: NEGATIVE | Score: 0.9997

Sentence: I enjoyed learning about transformers.
Label: POSITIVE | Score: 0.9997



## Task 3 — Encoder–Decoder Model

Use the FLAN-T5 model to perform the following:

- Summarize a short paragraph.


In [7]:
paragraph = (
    "A university library has extended its opening hours during final exams. "
    "It now stays open until midnight on weekdays so students have more time to study. "
    "The library also provides quiet study rooms and free tutoring sessions. "
    "These changes aim to help students prepare for their exams."
)

summary_prompt = "Summarize the following paragraph: " + paragraph
summary_inputs = tokenizer(
    summary_prompt, return_tensors="pt", truncation=True, max_length=512
)
with torch.inference_mode():
    summary_ids = model.generate(
        **summary_inputs,
        max_new_tokens=70,
        num_beams=4,
        do_sample=False,
        early_stopping=True,
    )
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print("Original paragraph:")
print(paragraph)
print("\nGenerated summary:")
print(summary)

Original paragraph:
A university library has extended its opening hours during final exams. It now stays open until midnight on weekdays so students have more time to study. The library also provides quiet study rooms and free tutoring sessions. These changes aim to help students prepare for their exams.

Generated summary:
The university library has extended its opening hours.



## Task 5 — Short Reflection

Answer in 3–5 sentences:

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**


I think a decoder-only model is suitable for a chatbot because it generates a response one token at a time using the conversation as context. Its causal attention allows each new token to depend on the earlier tokens in the prompt and response. This supports flexible, open-ended answers instead of choosing from a fixed set of labels. Instruction tuning and human feedback can help a pretrained model follow requests and produce more useful conversational responses.

# Optional Challenge

Try generating text several times with the same prompt.

Remove:

```python
set_seed(42)
```

and run the sampling cell multiple times.

### Question

Why do you sometimes get different outputs even when the prompt is the same?

**Hint:** Sampling introduces randomness when choosing the next token.


In [8]:
# I seed once in setup, but do not reset the seed between samples.
# Re-running this cell advances the random state and can change the results.
sampling_prompt = "The future of artificial intelligence is"
for attempt in range(1, 4):
    sample = generator(
        sampling_prompt,
        max_new_tokens=30,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        top_p=0.95,
        pad_token_id=generator.tokenizer.eos_token_id,
    )
    print(f"Sample {attempt}: {sample[0]['generated_text']}\n")

Sample 1: The future of artificial intelligence is still far from clear, but a lot of the data on these issues could be useful to the public.












Sample 2: The future of artificial intelligence is not yet known, but there are several possible applications. Among them is a new tool for building a neural network. It's more complex than most of



Sample 3: The future of artificial intelligence is under way in the 21st century, with many, many people and a lot of people wondering, "What is happening with artificial intelligence?" I think



### My explanation

I sometimes get different outputs because sampling randomly selects the next token from a probability distribution instead of always selecting the most likely token. Each selected token changes the context for later predictions, so the continuations can diverge. I set the seed once in setup for a repeatable sequence in the same environment, but I do not reset it between generations; removing that setup seed also allows separate runs to vary. Resetting the same seed before every generation usually reproduces the same result under the same conditions, while greedy decoding (`do_sample=False`) does not use this sampling randomness.

I kept the original task numbering: the supplied notebook goes directly from Task 3 to Task 5.